In [ ]:
import json
from pymongo import MongoClient, InsertOne
from datetime import datetime
import os

# ==========
# CONFIG
# ==========
MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "GreenCoop"
COLLECTION_NAME = "ObsProEtAmateur"
BACKUP_FILE = "data/backup_ObsProEtAmateur.jsonl"

TIME_FIELD = "dh_utc"
META_FIELD = "station_id"

# ==========
# Connexion
# ==========
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

print("✅ Connexion établie à MongoDB.")

# ==========
# Sauvegarde (avec conversion des datetime en chaîne)
# ==========
print("📦 Sauvegarde des documents...")

with open(BACKUP_FILE, "w", encoding="utf-8") as f_out:
    for doc in collection.find({}, {"_id": 0}):
        def convert(obj):
            if isinstance(obj, datetime):
                return obj.isoformat()
            return obj

        json.dump(doc, f_out, default=convert)
        f_out.write("\n")

print(f"✅ Sauvegarde effectuée → {BACKUP_FILE}")

# ==========
# Suppression de l'ancienne collection
# ==========
collection.drop()
print("🗑️ Collection supprimée")

# ==========
# Création de la nouvelle Time Series
# ==========
db.create_collection(
    COLLECTION_NAME,
    timeseries={
        "timeField": TIME_FIELD,
        "metaField": META_FIELD,
        "granularity": "hours"
    }
)
print("🆕 Nouvelle collection Time Series créée.")

# ==========
# Réimport avec conversion des dates ISO → datetime
# ==========
new_collection = db[COLLECTION_NAME]

print("📥 Réimport des données...")

ops = []
with open(BACKUP_FILE, "r", encoding="utf-8") as f_in:
    for line in f_in:
        doc = json.loads(line)

        # Reconvertir les dates ISO 8601 en objets datetime
        if TIME_FIELD in doc:
            try:
                doc[TIME_FIELD] = datetime.fromisoformat(doc[TIME_FIELD])
            except ValueError:
                print("⚠️ Format date invalide :", doc[TIME_FIELD])
                continue

        # Ignorer les documents sans metaField ou timeField
        if META_FIELD not in doc or TIME_FIELD not in doc:
            continue

        ops.append(InsertOne(doc))

# Insertion en batch
BATCH_SIZE = 1000
for i in range(0, len(ops), BATCH_SIZE):
    new_collection.bulk_write(ops[i:i + BATCH_SIZE])

print(f"✅ Réimport terminé ({len(ops)} documents insérés)")

# ==========
# Vérification
# ==========
print("🔍 Vérification :")
print("📄 Total documents :", new_collection.count_documents({}))
sample = new_collection.find_one()
print("📝 Exemple de document :")
print(json.dumps(sample, indent=2, default=str))

print("\n🎉 Migration vers Time Series AVEC metaField réussie.")


Voici un **script Jupyter Notebook complet** qui montre **comment effectuer toutes les opérations CRUD** (Create, Read, Update, Delete) sur ta **base MongoDB locale**.

Ce script fonctionne pour n’importe quelle base, mais je vais l’adapter à **ta base `GreenCoop` et la collection `ObsProEtAmateur`**.
Tu peux l’adapter facilement pour d’autres bases/collections.

---

### 📘 Prérequis

* MongoDB en local (sur `localhost:27017`)
* Base : `GreenCoop`
* Collection : `ObsProEtAmateur`
* Librairie : `pymongo` et `pandas`

---

## 🧪 CRUD – Script Complet avec Explication

```python
from pymongo import MongoClient
import pandas as pd
from bson.json_util import dumps

# Connexion au cluster MongoDB local
client = MongoClient("mongodb://localhost:27017/")
db = client["GreenCoop"]
collection = db["ObsProEtAmateur"]

# =======================
# 🔁 1. CREATE – Insérer des documents
# =======================
doc1 = {
    "station_name": "StationTest1",
    "ville": "Lille",
    "température_°C": 21.5,
    "dh_utc": pd.Timestamp("2024-10-01 12:00:00").to_pydatetime()
}

doc2 = {
    "station_name": "StationTest2",
    "ville": "Lyon",
    "température_°C": 19.2,
    "dh_utc": pd.Timestamp("2024-10-01 13:00:00").to_pydatetime()
}

# Insérer un seul document
inserted_id = collection.insert_one(doc1).inserted_id
print(f"✅ Document inséré avec ID : {inserted_id}")

# Insérer plusieurs documents
collection.insert_many([doc2])
print("✅ Plusieurs documents insérés.")

# =======================
# 🔍 2. READ – Lire les données
# =======================
# Lire tous les documents
print("\n📄 Tous les documents existants :")
for doc in collection.find().limit(5):
    print(doc)

# Lire avec un filtre
print("\n📄 Températures à Lille :")
for doc in collection.find({"ville": "Lille"}):
    print(doc)

# Lire dans un DataFrame Pandas
df = pd.DataFrame(list(collection.find({"ville": "Lille"})))
display(df)

# =======================
# ✏️ 3. UPDATE – Modifier un document
# =======================
# Mettre à jour la température à Lille à 22.0°C
result = collection.update_many(
    {"ville": "Lille"},
    {"$set": {"température_°C": 22.0}}
)
print(f"\n✏️ Documents modifiés : {result.modified_count}")

# =======================
# 🗑️ 4. DELETE – Supprimer des documents
# =======================
# Supprimer les documents avec station_name = StationTest1
deleted = collection.delete_many({"station_name": "StationTest1"})
print(f"\n🗑️ Documents supprimés : {deleted.deleted_count}")
```

---

## ✅ Ce que ce script  permet de faire :

| Action     | Opération                     | Ligne concernée |
| ---------- | ----------------------------- | --------------- |
| 🔁 CREATE  | `insert_one` / `insert_many`  | Sections 1      |
| 🔍 READ    | `find()` avec ou sans filtre  | Section 2       |
| ✏️ UPDATE  | `update_many` ou `update_one` | Section 3       |
| 🗑️ DELETE | `delete_many` ou `delete_one` | Section 4       |




In [5]:
from pymongo import MongoClient
import pandas as pd

# Connexion
client = MongoClient("mongodb://localhost:27017/")
db = client["GreenCoop"]
collection = db["ObsProEtAmateur"]

# =======================
# CREATE (autorisé)
# =======================
doc1 = {
    "station_id": "TEST_001",
    "station_name": "StationTest1",
    "ville": "Lille",
    "température_°C": 21.5,
    "dh_utc": pd.Timestamp("2024-10-01 12:00:00").to_pydatetime()
}

doc2 = {
    "station_id": "TEST_002",
    "station_name": "StationTest2",
    "ville": "Lyon",
    "température_°C": 19.2,
    "dh_utc": pd.Timestamp("2024-10-01 13:00:00").to_pydatetime()
}

collection.insert_many([doc1, doc2])
print("✅ INSERT OK")

# =======================
# READ (libre)
# =======================
print("\n📄 Données Lille :")
for d in collection.find({"ville": "Lille"}):
    print(d)

# =======================
# UPDATE (UNIQUEMENT metaField)
# =======================
# Exemple pédagogique : correction d’un station_id
result = collection.update_many(
    {"station_id": "TEST_001"},
    {"$set": {"station_id": "TEST_001_CORRIGE"}}
)

print(f"\n✏️ MetaField mis à jour : {result.modified_count}")

# =======================
# DELETE (metaField uniquement)
# =======================
deleted = collection.delete_many({
    "station_id": {"$in": ["TEST_001_CORRIGE", "TEST_002"]}
})

print(f"\n🗑️ Documents supprimés : {deleted.deleted_count}")


✅ INSERT OK

📄 Données Lille :
{'dh_utc': datetime.datetime(2024, 10, 1, 12, 0), 'station_name': 'StationTest1', 'ville': 'Lille', 'température_°C': 21.5, '_id': ObjectId('6960beef78590d364f386248')}
{'dh_utc': datetime.datetime(2024, 10, 1, 12, 0), 'station_id': 'TEST_001', 'station_name': 'StationTest1', '_id': ObjectId('6960c1ba78590d364f38624b'), 'ville': 'Lille', 'température_°C': 21.5}
{'dh_utc': datetime.datetime(2024, 10, 1, 12, 0), 'station_id': 'TEST_001', 'station_name': 'StationTest1', '_id': ObjectId('6960c2f278590d364f38624e'), 'ville': 'Lille', 'température_°C': 21.5}
{'dh_utc': datetime.datetime(2024, 10, 1, 12, 0), 'station_id': 'TEST_001', 'station_name': 'StationTest1', '_id': ObjectId('6960c3b078590d364f386251'), 'ville': 'Lille', 'température_°C': 21.5}
{'dh_utc': datetime.datetime(2024, 10, 5, 0, 0), 'station_id': '07015', 'source': 'Meteo-France via infoclimat.fr', '_id': ObjectId('6960bea178590d364f385dd0'), 'rayonnement_solaire_W/m²': None, 'elevation': 47, 'pr

Voici tout ce qu’il te faut pour créer, utiliser, et documenter un **module Python générique** pour effectuer des **CRUD Time Series** sécurisés avec MongoDB.

---

## 📦 Module Python `ts_crud.py`

📁 **Chemin suggéré :** place ce fichier à la racine de ton projet (ex : `GreenCoop/ts_crud.py`)
💡 Tu pourras ensuite l'importer dans tes notebooks Jupyter avec `import ts_crud`

```python
# ts_crud.py
from pymongo import MongoClient
from datetime import datetime
import pandas as pd

class TimeSeriesManager:
    def __init__(self, db_name="GreenCoop", collection_name="ObsProEtAmateur", uri="mongodb://localhost:27017/"):
        self.client = MongoClient(uri)
        self.db = self.client[db_name]
        self.collection = self.db[collection_name]
        print(f"✅ Connecté à {db_name}.{collection_name}")

    # ✅ Créer un ou plusieurs documents
    def insert(self, docs):
        if isinstance(docs, dict):
            result = self.collection.insert_one(docs)
            return result.inserted_id
        elif isinstance(docs, list):
            result = self.collection.insert_many(docs)
            return result.inserted_ids
        else:
            raise ValueError("❌ Format non reconnu pour l'insertion.")

    # 🔍 Lire les documents avec un filtre
    def read(self, filter_query={}, limit=10):
        return list(self.collection.find(filter_query).limit(limit))

    # 📝 Corriger une mesure = Delete + Insert
    def correct_measure(self, filter_query, new_doc):
        deleted = self.collection.delete_many(filter_query)
        inserted = self.collection.insert_one(new_doc)
        print(f"✅ Correction : {deleted.deleted_count} supprimé(s), 1 inséré.")
        return inserted.inserted_id

    # 🗑️ Supprimer par station_id
    def delete_by_station(self, station_id):
        deleted = self.collection.delete_many({"station_id": station_id})
        print(f"🗑️ {deleted.deleted_count} document(s) supprimé(s) pour station_id = {station_id}")
        return deleted.deleted_count

    # 📊 Charger les données dans un DataFrame
    def to_dataframe(self, filter_query={}):
        data = list(self.collection.find(filter_query))
        if not data:
            print("⚠️ Aucun document trouvé.")
            return pd.DataFrame()
        df = pd.DataFrame(data)
        return df
```

---

## 📓 Comment utiliser ce module dans un Jupyter Notebook

1. Place le fichier `ts_crud.py` **dans le même dossier que ton notebook**
2. Ensuite, dans une cellule Jupyter :

```python
# 🔁 Recharger automatiquement le module si modifié
%load_ext autoreload
%autoreload 2

# 📥 Importer
from ts_crud import TimeSeriesManager

# 🚀 Initialiser le gestionnaire
ts = TimeSeriesManager()

# ✅ Exemple 1 : insertion
ts.insert({
    "station_id": "TEST_LILLE",
    "dh_utc": pd.Timestamp("2024-10-01 14:00:00"),
    "ville": "Lille",
    "température_°C": 18.5
})

# 🔍 Exemple 2 : lecture
df = ts.to_dataframe({"ville": "Lille"})
df.head()

# ✏️ Exemple 3 : correction
ts.correct_measure(
    filter_query={"station_id": "TEST_LILLE", "dh_utc": pd.Timestamp("2024-10-01 14:00:00")},
    new_doc={
        "station_id": "TEST_LILLE",
        "dh_utc": pd.Timestamp("2024-10-01 14:00:00"),
        "ville": "Lille",
        "température_°C": 19.8  # corrigée
    }
)

# 🗑️ Exemple 4 : suppression
ts.delete_by_station("TEST_LILLE")
```

---

## 📌 Pourquoi c’est important (rappel Time Series MongoDB)

* MongoDB **n'autorise pas de UPDATE** sur les champs de mesure dans une collection Time Series.
* Toute **correction de valeur** doit se faire en 2 étapes :

  * `DELETE` (avec un filtre sur `station_id`)
  * puis `INSERT` du nouveau document

Ce module t’apporte :

* ✅ Une interface claire
* ✅ Des méthodes sûres
* ✅ Zéro erreur MongoDB (`WriteError`)

---

Souhaites-tu que je te fournisse une version `.py` téléchargeable directement ?
